<a href="https://colab.research.google.com/github/RonYehuda/mnist-mlp-pytorch/blob/main/MNIST_MLP_full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST MLP - full pipeline

## 1. Imports

In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd


## 2. Load and normalize the MNIST training data

In [2]:
train_df = pd.read_csv("/content/mnist_train.csv")

labels_all = torch.tensor(train_df.iloc[:, 0].values, dtype=torch.int64)
# dividing by 255.0 (a float) upcasts the whole tensor to float32 automatically
images_all = torch.tensor(train_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0

print(images_all.shape, labels_all.shape)  # expected: [60000, 784] [60000]


torch.Size([60000, 784]) torch.Size([60000])


## 3. Split into train / validation sets
Splitting off 10,000 samples for validation, before any training happens.

In [3]:
n_total = len(images_all)
n_val = 10000

# random permutation decides WHICH samples go to validation, once, up front
shuffled = torch.randperm(n_total)
val_idx, train_idx = shuffled[:n_val], shuffled[n_val:]

val_images, val_labels = images_all[val_idx], labels_all[val_idx]
train_images, train_labels = images_all[train_idx], labels_all[train_idx]

print(train_images.shape, val_images.shape)  # expected: [50000, 784] [10000, 784]


torch.Size([50000, 784]) torch.Size([10000, 784])


## 4. Model definition (MLP with BatchNorm + Dropout)

In [4]:
class MLP(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.bn1 = nn.BatchNorm1d(num_features=128)   # must match fc1's output size
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(num_features=64)    # must match fc2's output size
        self.fc3 = nn.Linear(64, 10)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        x = x.view(-1, 784)

        # order matters here: Linear -> BatchNorm -> ReLU -> Dropout
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.activation(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.activation(x)
        x = self.dropout(x)

        x = self.fc3(x)  # no activation/dropout on the final layer: raw logits
        return x


## 5. Custom Dataset class
Wraps the tensors so a `DataLoader` can use them.

In [5]:
class MNISTDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels):
        super().__init__()
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


## 6. Datasets and DataLoaders

In [6]:
train_dataset = MNISTDataset(train_images, train_labels)
val_dataset = MNISTDataset(val_images, val_labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)   # shuffle every epoch
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)        # no need to shuffle validation

# sanity check
batch_images, batch_labels = next(iter(train_loader))
print(batch_images.shape, batch_labels.shape)  # expected: [64, 784] [64]


torch.Size([64, 784]) torch.Size([64])


## 7. Accuracy helper function
Used for both train accuracy and validation accuracy, on any `DataLoader`.

In [7]:
def model_accuracy(model, data_loader):
    model.eval()  # disables Dropout and switches BatchNorm to running statistics
    with torch.no_grad():
        n_correct = 0
        n_total = 0
        for images, labels in data_loader:
            preds = model(images).argmax(dim=1)
            n_correct += (labels == preds).sum().item()
            n_total += len(labels)
    return n_correct / n_total


## 8. Training loop (with gradient clipping, LR scheduler, checkpoints)

In [8]:
def training_loop(model, train_loader, val_loader, criterion, optimizer, n_epochs=10,
                   checkpoint_path=None, scheduler=None):

    if checkpoint_path is not None and os.path.isfile(checkpoint_path):
        checkpoint_data = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint_data["model_state_dict"])
        optimizer.load_state_dict(checkpoint_data["optimizer_state_dict"])
        # restore the scheduler too, otherwise the LR schedule restarts from epoch 0 on resume
        if scheduler is not None and "scheduler_state_dict" in checkpoint_data:
            scheduler.load_state_dict(checkpoint_data["scheduler_state_dict"])
        start_epoch = checkpoint_data["epoch"] + 1
    else:
        start_epoch = 0

    for epoch in range(start_epoch, n_epochs):
        model.train()  # enables Dropout and per-batch BatchNorm statistics
        current_loss = 0

        for batch_images, batch_labels in train_loader:
            pred = model(batch_images)
            loss = criterion(pred, batch_labels)
            loss.backward()
            # clip gradients BEFORE the optimizer step, to avoid unstable/huge updates
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            current_loss += loss.item() * len(batch_images)

        # scheduler.step() is called once per epoch, not once per batch
        if scheduler is not None:
            scheduler.step()

        train_accuracy = model_accuracy(model, train_loader)
        val_accuracy = model_accuracy(model, val_loader)
        print(f"epoch {epoch+1}/{n_epochs}  loss: {current_loss/len(train_loader.dataset):.4f}  "
              f"train_acc: {train_accuracy:.4f}  val_acc: {val_accuracy:.4f}")

        if checkpoint_path is not None:
            checkpoint_data = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            }
            if scheduler is not None:
                checkpoint_data["scheduler_state_dict"] = scheduler.state_dict()
            torch.save(checkpoint_data, checkpoint_path)


## 9. Run training

In [9]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)  # halves lr every 5 epochs

training_loop(model, train_loader, val_loader, criterion, optimizer,
              n_epochs=10, checkpoint_path="checkpoint.pt", scheduler=scheduler)


epoch 1/10  loss: 0.6207  train_acc: 0.9416  val_acc: 0.9419
epoch 2/10  loss: 0.3656  train_acc: 0.9571  val_acc: 0.9527
epoch 3/10  loss: 0.3119  train_acc: 0.9626  val_acc: 0.9554
epoch 4/10  loss: 0.2772  train_acc: 0.9690  val_acc: 0.9615
epoch 5/10  loss: 0.2589  train_acc: 0.9724  val_acc: 0.9629
epoch 6/10  loss: 0.2340  train_acc: 0.9755  val_acc: 0.9672
epoch 7/10  loss: 0.2168  train_acc: 0.9776  val_acc: 0.9684
epoch 8/10  loss: 0.2098  train_acc: 0.9778  val_acc: 0.9692
epoch 9/10  loss: 0.2095  train_acc: 0.9796  val_acc: 0.9701
epoch 10/10  loss: 0.2061  train_acc: 0.9802  val_acc: 0.9688


## 10. Load and normalize the test set
Used only once, at the very end, never during training or tuning.

In [10]:
test_df = pd.read_csv("/content/mnist_test.csv")

test_labels = torch.tensor(test_df.iloc[:, 0].values, dtype=torch.int64)
# same dtype and same /255.0 normalization as the training data, this must match exactly
test_images = torch.tensor(test_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0

print(test_images.shape, test_labels.shape)  # expected: [10000, 784] [10000]


torch.Size([10000, 784]) torch.Size([10000])


## 11. Test dataset and DataLoader

In [11]:
test_dataset = MNISTDataset(test_images, test_labels)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)  # no need to shuffle


## 12. Evaluate on the test set

In [12]:
test_accuracy = model_accuracy(model, test_loader)
print(f"test accuracy: {test_accuracy:.4f}")


test accuracy: 0.9714
